# 04 Reproduce H→bb baseline results (working notebook)

This notebook reproduces the **same H→bb/STXS χ² logic** used in `stxs_functions.py` and visualized in `fit_plots.ipynb`, but in a robust way that works from your cloned repo path.

## Critical fix vs previous broken version
`sf.stxs_fit(coffea_name, ...)` expects `coffea/<name>.coffea` relative to current working directory and also relies on internal hard-coded paths.

This notebook avoids that fragility by:
1. Loading coffea by **full path** from `asset_manifest.json`.
2. Reusing the same mathematical model from `stxs_functions.py` (quadratic fit + χ² definition).
3. Producing outputs/plots in the same style as `fit_plots.ipynb`.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt, mplhep as hep
from matplotlib.lines import Line2D
from coffea import util
from scipy.optimize import curve_fit
hep.style.use('CMS')

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p=Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p/marker).exists(): return p
        if p.parent==p: break
        p=p.parent
    raise FileNotFoundError(f'Could not find {marker} from {Path.cwd()}')

REPO=find_repo_root()
NBDIR=REPO/'notebooks_hww_fit'
manifest=json.loads((NBDIR/'asset_manifest.json').read_text())
coffea_fullpath=manifest['default_coffea_fullpath']
print('repo:',REPO)
print('coffea_fullpath:',coffea_fullpath)

In [ ]:
# Import stxs_functions for helper pieces (wc map and bin-yield extraction).
for pth in ['/uscms_data/d3/azhou/smeft/analysis','/uscms_data/d3/azhou/smeft/analysis/hbb-coffea']:
    if pth not in sys.path: sys.path.append(pth)
import stxs_functions as sf

# Operator list used in legacy fit_plots.ipynb
SMEFT_OPS=['cHbox','cHDD','cHj1','cHj3','cHu','cHd','cHudRe','cuWRe','cuBRe','cdWRe','cdBRe','cHW','cHB','cHWB']
SMEFTCPV_OPS=['cHudIm','cuWIm','cdWIm','cuBIm','cdBIm','cHWtil','cHBtil','cHWBtil']
ALL_OPS=SMEFT_OPS+SMEFTCPV_OPS
print('n operators:',len(ALL_OPS))

In [ ]:
# ---- Exact Hbb chi2 model from stxs_functions.py ----
def chisq_stxs_hbb(calculated_xsec1, calculated_xsec2):
    cms_obs = np.array([240., 120., 200., 190., 68., 61.])
    bin1_obs, bin2_obs = cms_obs[0], cms_obs[1]
    sig1_up, sig1_down, sig2_up, sig2_down = cms_obs[2], cms_obs[3], cms_obs[4], cms_obs[5]

    bin1_diff = calculated_xsec1 - bin1_obs
    bin2_diff = calculated_xsec2 - bin2_obs
    sigma1 = sig1_up if bin1_diff >= 0 else sig1_down
    sigma2 = sig2_up if bin2_diff >= 0 else sig2_down

    bin1_chisq = ((calculated_xsec1 - bin1_obs)/sigma1)**2
    bin2_chisq = ((calculated_xsec2 - bin2_obs)/sigma2)**2
    total_chisq = bin1_chisq + bin2_chisq
    return bin1_chisq, bin2_chisq, total_chisq

def stxs_reweight_coeffs_fullpath(coffea_path, operator_list):
    all_h = util.load(coffea_path)['htxs']
    wc_mapping = sf.wc_map_dict(operator_list)

    bin1_yield, bin2_yield = {}, {}
    for p,wc_label in wc_mapping.items():
        if wc_label not in list(all_h.axes['wc']):
            continue
        hslice = all_h[{'wc': wc_label}]
        b1,b2 = sf.get_bin_yields(hslice)
        bin1_yield[p]=b1
        bin2_yield[p]=b2

    pts=[p for p in wc_mapping.keys() if p in bin1_yield and p in bin2_yield]
    if len(operator_list)==1:
        x=np.array([p[0] for p in pts])
        y1=np.array([bin1_yield[p] for p in pts]); y2=np.array([bin2_yield[p] for p in pts])
        c1,_=curve_fit(sf.quad_1D, x, y1, p0=[1,1,1])
        c2,_=curve_fit(sf.quad_1D, x, y2, p0=[1,1,1])
    elif len(operator_list)==2:
        x1=np.array([p[0] for p in pts]); x2=np.array([p[1] for p in pts])
        y1=np.array([bin1_yield[p] for p in pts]); y2=np.array([bin2_yield[p] for p in pts])
        c1,_=curve_fit(sf.quad_2D, (x1,x2), y1, p0=[1,1,1,1,1,1])
        c2,_=curve_fit(sf.quad_2D, (x1,x2), y2, p0=[1,1,1,1,1,1])
    else:
        raise ValueError('Only 1D/2D supported')
    return c1,c2

In [ ]:
def hbb_stxs_fit_fullpath(coffea_path, operator_list, wc_min=-100, wc_max=100, n=500, mg_sigma_pb=3.594):
    if len(operator_list)>2:
        raise ValueError('Only <=2 operators supported, matching legacy stxs_fit')

    c1,c2 = stxs_reweight_coeffs_fullpath(coffea_path, operator_list)
    sumw = util.load(coffea_path)['sumw_all_noEW'].value

    wc_space_dict={}
    if len(operator_list)==1:
        for x in np.linspace(wc_min,wc_max,n):
            b1 = mg_sigma_pb*1000*sf.quad_1D(x,*c1)/sumw
            b2 = mg_sigma_pb*1000*sf.quad_1D(x,*c2)/sumw
            bt1,bt2,btt = chisq_stxs_hbb(b1,b2)
            wc_space_dict[float(x)]={'bin1':[float(b1),float(bt1)],'bin2':[float(b2),float(bt2)],'total':[float(b1+b2),float(btt)]}
    else:
        g=np.linspace(wc_min,wc_max,n)
        for x in g:
            for y in g:
                b1 = mg_sigma_pb*1000*sf.quad_2D((x,y),*c1)/sumw
                b2 = mg_sigma_pb*1000*sf.quad_2D((x,y),*c2)/sumw
                bt1,bt2,btt = chisq_stxs_hbb(b1,b2)
                wc_space_dict[(float(x),float(y))]={'bin1':[float(b1),float(bt1)],'bin2':[float(b2),float(bt2)],'total':[float(b1+b2),float(btt)]}
    return wc_space_dict

In [ ]:
def hbb_fit_1d(op, coffea_path=coffea_fullpath, wc_min=-100, wc_max=100, n=500):
    ws = hbb_stxs_fit_fullpath(coffea_path, [op], wc_min=wc_min, wc_max=wc_max, n=n)
    x=np.array(sorted(ws.keys()),dtype=float)
    y=np.array([ws[p]['total'][1] for p in x],dtype=float)
    return pd.DataFrame({'op':op,'wc':x,'chi2_hbb':y}), ws

def run_all_hbb_1d(ops=ALL_OPS, **scan_kwargs):
    out_df, out_ws = {}, {}
    for op in ops:
        try:
            d,w = hbb_fit_1d(op, **scan_kwargs)
            out_df[op]=d; out_ws[op]=w
        except Exception as e:
            print('skip',op,e)
    return out_df, out_ws

In [ ]:
def plot_hbb_1d(df):
    plt.figure(figsize=(6,4))
    plt.plot(df['wc'], df['chi2_hbb'], label=df['op'].iloc[0])
    cmin=df['chi2_hbb'].min()
    plt.axhline(cmin+1, color='tab:red', ls='--', label='1$\sigma$')
    plt.xlabel('Wilson coefficient value')
    plt.ylabel(r'$\chi^2$')
    plt.xlim(-10,10)
    plt.ylim(0,10)
    plt.title(f"Hbb 1D $\chi^2$ scan: {df['op'].iloc[0]}")
    plt.legend()
    plt.tight_layout()

In [ ]:
# Quick validation against legacy style: single-operator and 2D contour example
# df_chw, ws_chw = hbb_fit_1d('cHW', wc_min=-10, wc_max=10, n=401)
# plot_hbb_1d(df_chw)

# ws_2d = hbb_stxs_fit_fullpath(coffea_fullpath, ['cHW','cHWtil'], wc_min=-10, wc_max=10, n=121)
# (Optional) convert ws_2d to heatmap in same style as fit_plots.ipynb

In [ ]:
# Batch run all 1D Hbb fits (can take time)
# HBB_ALL_DF, HBB_ALL_WS = run_all_hbb_1d(wc_min=-10, wc_max=10, n=401)
# summary = pd.DataFrame([{'op':op,'best_wc':d.loc[d.chi2_hbb.idxmin(),'wc'],'chi2_min':d['chi2_hbb'].min()} for op,d in HBB_ALL_DF.items()]).sort_values('chi2_min')
# summary.head(20)